1. Struktur Dataset

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

PATH_DATA = "dataset_hipertensi_prepared.csv"
OUTPUT_DIR = "EDA_After_Preparation_Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def garis(judul):
    print("\n" + "="*70)
    print(judul)
    print("="*70)


def simpan(nama):
    filepath = os.path.join(OUTPUT_DIR, nama)

    plt.tight_layout()
    plt.savefig(filepath, dpi=120, bbox_inches="tight")
    plt.close()

    print(f"[Saved] {filepath}")


df = pd.read_csv(PATH_DATA)

garis("A. DATASET OVERVIEW")

print("Shape :", df.shape)
print("\nData Type")
print(df.dtypes)

print("\nMissing Value (%)")
print((df.isna().mean()*100).round(2))

print("\nDuplicate Rows :", df.duplicated().sum())


A. DATASET OVERVIEW
Shape : (32541, 9)

Data Type
age                     float64
is_female               float64
bmi                     float64
is_smoker               float64
has_diabetes            float64
has_high_cholesterol    float64
sleep_quality           float64
sleep_disturbance       float64
label_hypertension        int64
dtype: object

Missing Value (%)
age                     0.02
is_female               0.02
bmi                     0.62
is_smoker               0.48
has_diabetes            0.53
has_high_cholesterol    0.53
sleep_quality           4.88
sleep_disturbance       4.88
label_hypertension      0.00
dtype: float64

Duplicate Rows : 24


2. Distribusi Target

In [2]:
garis("B. TARGET DISTRIBUTION")

target = df["label_hypertension"].value_counts().sort_index()

print(target)

plt.figure(figsize=(6,4))

bars = plt.bar(
    ["Non-Hypertension","Hypertension"],
    target.values,
    color=["steelblue","indianred"]
)

for b,v in zip(bars,target.values):
    plt.text(
        b.get_x()+b.get_width()/2,
        v,
        f"{v}\n({v/len(df):.1%})",
        ha="center"
    )

plt.ylabel("Number of Respondents")
plt.title("Distribution of Hypertension Label")

simpan("g1_target_distribution.png")


B. TARGET DISTRIBUTION
label_hypertension
0    23491
1     9050
Name: count, dtype: int64
[Saved] EDA_After_Preparation_Output\g1_target_distribution.png


3. Statistik Deskriptif

In [3]:
garis("C. DESCRIPTIVE STATISTICS")

numeric = [
    "age",
    "bmi",
    "sleep_quality",
    "sleep_disturbance"
]

print(df[numeric].describe().round(2))


C. DESCRIPTIVE STATISTICS
            age       bmi  sleep_quality  sleep_disturbance
count  32534.00  32340.00       30954.00           30954.00
mean      38.26     23.19           3.29               1.95
std       15.90      4.48           0.82               1.17
min       15.00     10.85           1.00               1.00
25%       26.00     19.86           3.00               1.00
50%       35.00     22.56           3.00               1.00
75%       48.00     25.89           4.00               3.00
max      110.00     68.76           5.00               5.00


4. Distribusi Variabel Numerik

In [4]:
garis("D. NUMERICAL FEATURE DISTRIBUTION")

for col in numeric:

    plt.figure(figsize=(6,4))

    plt.hist(
        df[col],
        bins=30,
        edgecolor="black"
    )

    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Frequency")

    simpan(f"hist_{col}.png")


D. NUMERICAL FEATURE DISTRIBUTION
[Saved] EDA_After_Preparation_Output\hist_age.png
[Saved] EDA_After_Preparation_Output\hist_bmi.png
[Saved] EDA_After_Preparation_Output\hist_sleep_quality.png
[Saved] EDA_After_Preparation_Output\hist_sleep_disturbance.png


5. Distribusi Variabel Kategorikal

In [5]:
garis("E. CATEGORICAL FEATURES")

binary = [
    "is_female",
    "is_smoker",
    "has_diabetes",
    "has_high_cholesterol"
]

for col in binary:

    vc = df[col].value_counts().sort_index()

    print(f"\n{col}")
    print(vc)

    plt.figure(figsize=(5,4))

    bars = plt.bar(
        vc.index.astype(str),
        vc.values
    )

    for b,v in zip(bars,vc.values):
        plt.text(
            b.get_x()+b.get_width()/2,
            v,
            str(v),
            ha="center"
        )

    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Count")

    simpan(f"{col}.png")


E. CATEGORICAL FEATURES

is_female
is_female
0.0    15230
1.0    17305
Name: count, dtype: int64
[Saved] EDA_After_Preparation_Output\is_female.png

is_smoker
is_smoker
0.0    20681
1.0    11705
Name: count, dtype: int64
[Saved] EDA_After_Preparation_Output\is_smoker.png

has_diabetes
has_diabetes
0.0    31659
1.0      711
Name: count, dtype: int64
[Saved] EDA_After_Preparation_Output\has_diabetes.png

has_high_cholesterol
has_high_cholesterol
0.0    31062
1.0     1308
Name: count, dtype: int64
[Saved] EDA_After_Preparation_Output\has_high_cholesterol.png


6. Korelasi Antar Variabel

In [6]:
garis("F. CORRELATION MATRIX")

plt.figure(figsize=(8,6))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

simpan("g2_correlation.png")


F. CORRELATION MATRIX
[Saved] EDA_After_Preparation_Output\g2_correlation.png


7. Hubungan Fitur terhadap Hipertensi

In [7]:
garis("G. NUMERICAL FEATURES VS TARGET")

for col in numeric:

    plt.figure(figsize=(6,4))

    sns.boxplot(
        x="label_hypertension",
        y=col,
        data=df
    )

    plt.title(f"{col} vs Hypertension")

    simpan(f"{col}_vs_target.png")
    
#     untuk variabel biner

garis("H. BINARY FEATURES VS TARGET")

for col in binary:

    rate = (
        df.groupby(col)["label_hypertension"]
        .mean()*100
    )

    plt.figure(figsize=(5,4))

    bars = plt.bar(
        ["No","Yes"],
        rate.values,
        color=["steelblue","indianred"]
    )

    for b,v in zip(bars,rate.values):
        plt.text(
            b.get_x()+b.get_width()/2,
            v,
            f"{v:.1f}%",
            ha="center"
        )

    plt.ylabel("Hypertension (%)")
    plt.title(col)

    simpan(f"{col}_target_rate.png")


G. NUMERICAL FEATURES VS TARGET
[Saved] EDA_After_Preparation_Output\age_vs_target.png
[Saved] EDA_After_Preparation_Output\bmi_vs_target.png
[Saved] EDA_After_Preparation_Output\sleep_quality_vs_target.png
[Saved] EDA_After_Preparation_Output\sleep_disturbance_vs_target.png

H. BINARY FEATURES VS TARGET
[Saved] EDA_After_Preparation_Output\is_female_target_rate.png
[Saved] EDA_After_Preparation_Output\is_smoker_target_rate.png
[Saved] EDA_After_Preparation_Output\has_diabetes_target_rate.png
[Saved] EDA_After_Preparation_Output\has_high_cholesterol_target_rate.png


8. Feature Importance Sederhana 

In [8]:
garis("I. FEATURE CORRELATION WITH TARGET")

corr = (
    df.corr(numeric_only=True)["label_hypertension"]
    .drop("label_hypertension")
    .sort_values(key=np.abs, ascending=False)
)

print(corr)

plt.figure(figsize=(7,4))

corr.plot.bar()

plt.ylabel("Correlation")

plt.title("Feature Correlation with Hypertension")

simpan("g3_feature_target_corr.png")


I. FEATURE CORRELATION WITH TARGET
age                     0.415652
bmi                     0.204170
has_high_cholesterol    0.111450
has_diabetes            0.096065
sleep_quality           0.050949
sleep_disturbance      -0.027046
is_smoker               0.017727
is_female              -0.002612
Name: label_hypertension, dtype: float64
[Saved] EDA_After_Preparation_Output\g3_feature_target_corr.png
